# Social-STGCNN -- Collision-Aware Model (hinge loss)

Trains, evaluates, and visualises Social-STGCNN with the **hinge** collision-aware auxiliary loss on **UNIV** and **ZARA2**.

| Step | Section |
|------|---------|
| 1 | Environment |
| 2 | **Configuration** (only cell to edit) |
| 3 | Get the code |
| 4 | Train (UNIV + ZARA2 only) |
| 5 | Evaluate ADE / FDE / ColRate |
| 6 | Results table vs baseline |
| 7 | Collision bar chart |
| 8 | Trajectory visualization |
| 9 | CSV export |


## 1. Environment

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Configuration

**Only this cell needs editing between runs.**

Available losses: `hinge` | `exp` | `inv` | `gauss` | `ecp`

In [ ]:
# -- Edit this block only ------------------------------------------
COLLISION_LOSS = 'hinge'   # hinge | exp | inv | gauss | ecp
LAMBDA_COL     = 1.0       # weight for collision loss (0 disables it)
NUM_EPOCHS     = 250
# -------------------------------------------------------------------

DATASETS = ['univ', 'zara2']
SAVE_DIR = COLLISION_LOSS

print(f'Collision loss : {COLLISION_LOSS}')
print(f'Lambda col     : {LAMBDA_COL}')
print(f'Save directory : {SAVE_DIR}/')
print(f'Datasets       : {DATASETS}')
print(f'Epochs         : {NUM_EPOCHS}')

## 3. Get the code

Run whichever option applies -- skip the others.


In [ ]:
# Option A: Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# import os
# PROJECT_DIR = '/content/drive/MyDrive/DLproject-Social-STGCNN'
# assert os.path.isdir(PROJECT_DIR); os.chdir(PROJECT_DIR)
# print('Working directory:', os.getcwd())

In [ ]:
# Option B: GitHub clone
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git Social-STGCNN
# import os; os.chdir('Social-STGCNN')
# print('Working directory:', os.getcwd())

In [ ]:
# Option C: Local machine
import os
PROJECT_DIR = r'D:\OMSCS\DLproject-Social-STGCNN'
assert os.path.isdir(PROJECT_DIR), f'Repo not found at {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

## 4. Train -- UNIV + ZARA2

Saves to `hinge/social-stgcnn-{dataset}/val_best.pth`.  
Existing checkpoints are skipped automatically.

In [ ]:
import os, subprocess, sys

for ds in DATASETS:
    dst = f'{SAVE_DIR}/social-stgcnn-{ds}'
    if os.path.isfile(dst + '/val_best.pth'):
        print(f'[SKIP] {ds} -- checkpoint exists at {dst}/'); continue

    print(f'Training ({COLLISION_LOSS}, lambda={LAMBDA_COL}): {ds} -> {dst}/')

    result = subprocess.run([
        sys.executable, 'train.py',
        '--lr',             '0.01',
        '--n_stgcnn',       '1',
        '--n_txpcnn',       '5',
        '--dataset',        ds,
        '--tag',            f'social-stgcnn-{ds}',
        '--save_dir',       SAVE_DIR,
        '--use_lrschd',
        '--num_epochs',     str(NUM_EPOCHS),
        '--collision_loss', COLLISION_LOSS,
        '--lambda_col',     str(LAMBDA_COL),
    ])
    status = 'OK' if result.returncode == 0 else 'FAILED'
    print(f'[{status}] {ds} -> {dst}/')

print('\nCheckpoint status:')
for ds in DATASETS:
    dst = f'{SAVE_DIR}/social-stgcnn-{ds}'
    ok = os.path.isfile(dst + '/val_best.pth')
    print(f'  {ds:8s}  [{"OK" if ok else "MISSING"}]  {dst}/')

## 5. Evaluate -- ADE / FDE / ColRate

K=20 Monte-Carlo samples, same protocol as the baseline.


In [ ]:
import os, pickle, copy
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch, numpy as np
import torch.distributions.multivariate_normal as torchdist
from torch.utils.data import DataLoader
from tqdm.std import tqdm

from utils import TrajectoryDataset
from metrics import ade, fde, collision_metrics, seq_to_nodes, nodes_rel_to_nodes_abs
from model import social_stgcnn

KSTEPS = 20
D_COL  = 0.2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


def evaluate_checkpoint(exp_path):
    with open(exp_path + '/args.pkl', 'rb') as f:
        args = pickle.load(f)

    dset_test = TrajectoryDataset(
        f'./datasets/{args.dataset}/test/',
        obs_len=args.obs_seq_len, pred_len=args.pred_seq_len,
        skip=1, norm_lap_matr=True)
    loader_test = DataLoader(dset_test, batch_size=1, shuffle=False, num_workers=0)

    model = social_stgcnn(
        n_stgcnn=args.n_stgcnn, n_txpcnn=args.n_txpcnn,
        output_feat=args.output_size, seq_len=args.obs_seq_len,
        kernel_size=args.kernel_size, pred_seq_len=args.pred_seq_len
    ).to(device)
    model.load_state_dict(torch.load(exp_path + '/val_best.pth', map_location=device))
    model.eval()

    ade_bigls, fde_bigls = [], []
    col_avg_ls, col_minade_ls, scene_col_avg = [], [], []

    with torch.no_grad():
        for batch in loader_test:
            batch = [t.to(device) for t in batch]
            obs_traj, _, obs_traj_rel, _, _, _, V_obs, A_obs, V_tr, _ = batch
            N = obs_traj_rel.shape[1]

            V_pred, _ = model(V_obs.permute(0, 3, 1, 2), A_obs.squeeze())
            V_pred = V_pred.permute(0, 2, 3, 1).squeeze()
            V_tr   = V_tr.squeeze()
            V_pred, V_tr = V_pred[:, :N, :], V_tr[:, :N, :]

            sx   = torch.exp(V_pred[:, :, 2])
            sy   = torch.exp(V_pred[:, :, 3])
            corr = torch.tanh(V_pred[:, :, 4])
            cov  = torch.zeros(*V_pred.shape[:2], 2, 2, device=device)
            cov[:, :, 0, 0] = sx * sx
            cov[:, :, 0, 1] = corr * sx * sy
            cov[:, :, 1, 0] = corr * sx * sy
            cov[:, :, 1, 1] = sy * sy
            mvn = torchdist.MultivariateNormal(V_pred[:, :, 0:2], cov)

            V_x = seq_to_nodes(obs_traj.cpu().numpy().copy())
            V_y = nodes_rel_to_nodes_abs(
                V_tr.cpu().numpy().squeeze().copy(), V_x[-1, :, :].copy())

            pred_samples = []
            ade_ls = {n: [] for n in range(N)}
            fde_ls = {n: [] for n in range(N)}

            for _ in range(KSTEPS):
                s  = mvn.sample()
                sa = nodes_rel_to_nodes_abs(
                    s.cpu().numpy().squeeze().copy(), V_x[-1, :, :].copy())
                pred_samples.append(copy.deepcopy(sa))
                for n in range(N):
                    ade_ls[n].append(ade([sa[:, n:n+1, :]], [V_y[:, n:n+1, :]], [1]))
                    fde_ls[n].append(fde([sa[:, n:n+1, :]], [V_y[:, n:n+1, :]], [1]))

            for n in range(N):
                ade_bigls.append(min(ade_ls[n]))
                fde_bigls.append(min(fde_ls[n]))

            col_res = collision_metrics(
                [s[:, :N, :] for s in pred_samples], V_y[:, :N, :], d_col=D_COL)
            col_avg_ls.append(col_res['ColRate_avg'])
            col_minade_ls.append(col_res['ColRate_minADE'])
            scene_col_avg.append(col_res['ColRate_avg'])

    n = len(ade_bigls)
    return {
        'ade':                sum(ade_bigls) / n,
        'fde':                sum(fde_bigls) / n,
        'col_avg':            float(np.mean(col_avg_ls)),
        'col_minade':         float(np.mean(col_minade_ls)),
        'pct_scenes_any_col': float(np.mean([c > 0 for c in scene_col_avg])),
    }


results = {}
for ds in tqdm(DATASETS, desc='Evaluating'):
    exp_path = f'{SAVE_DIR}/social-stgcnn-{ds}'
    if not os.path.isfile(exp_path + '/val_best.pth'):
        print(f'  SKIP {ds} -- checkpoint not found'); continue
    print(f'  Evaluating {ds} ...')
    results[ds] = evaluate_checkpoint(exp_path)
    r = results[ds]
    print(f'    ADE={r["ade"]:.4f}  FDE={r["fde"]:.4f}  '
          f'ColRate_avg={r["col_avg"]:.4f}  ColRate_minADE={r["col_minade"]:.4f}')
print('\nDone.')

## 6. Results table

Compares against the NLL-only baseline and paper numbers.  
Baseline numbers loaded from `baseline_results.csv` if available.

In [ ]:
import pandas as pd

PAPER = {'univ': (0.44, 0.79), 'zara2': (0.30, 0.48)}

df_base = None
if os.path.isfile('baseline_results.csv'):
    df_base = pd.read_csv('baseline_results.csv').set_index('scene')
else:
    print('baseline_results.csv not found -- run baseline notebook first')

rows = []
for ds in DATASETS:
    if ds not in results: continue
    r      = results[ds]
    pa, pf = PAPER.get(ds, (None, None))
    b_ade  = round(df_base.loc[ds, 'ADE_ours'], 4)    if df_base is not None and ds in df_base.index else None
    b_fde  = round(df_base.loc[ds, 'FDE_ours'], 4)    if df_base is not None and ds in df_base.index else None
    b_col  = round(df_base.loc[ds, 'ColRate_avg'], 4) if df_base is not None and ds in df_base.index else None
    rows.append({
        'Scene':               ds,
        'ADE (paper)':         pa,
        'ADE (baseline)':      b_ade,
        f'ADE ({COLLISION_LOSS})':     round(r['ade'], 4),
        'FDE (paper)':         pf,
        'FDE (baseline)':      b_fde,
        f'FDE ({COLLISION_LOSS})':     round(r['fde'], 4),
        'ColRate (baseline)':  b_col,
        f'ColRate ({COLLISION_LOSS})': round(r['col_avg'], 4),
    })

df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False))
csv_out = f'{SAVE_DIR}/collision_results_{COLLISION_LOSS}.csv'
df_res.to_csv(csv_out, index=False)
print(f'\nSaved: {csv_out}')

## 7. Collision analysis

Grouped bar chart: baseline (red/orange) vs this model (blue/teal).  
If only blue/teal bars appear, run the baseline notebook to generate `baseline_results.csv`.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
import os
os.makedirs(SAVE_DIR, exist_ok=True)

scenes_with_data = [ds for ds in DATASETS if ds in results]
col_avg_ours    = [results[ds]['col_avg']    for ds in scenes_with_data]
col_minade_ours = [results[ds]['col_minade'] for ds in scenes_with_data]

col_avg_base, col_minade_base = None, None
if df_base is not None:
    col_avg_base    = [df_base.loc[ds, 'ColRate_avg']    if ds in df_base.index else 0 for ds in scenes_with_data]
    col_minade_base = [df_base.loc[ds, 'ColRate_minADE'] if ds in df_base.index else 0 for ds in scenes_with_data]

x = np.arange(len(scenes_with_data))
w = 0.20

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle(f'Collision Rates: Baseline (NLL) vs {COLLISION_LOSS} loss'
             f'  (d_col={D_COL} m, K={KSTEPS})',
             fontsize=11, fontweight='bold')

if col_avg_base is not None:
    ax.bar(x - 1.5*w, col_avg_base,    w, label='ColRate_avg (baseline)',          color='#e07070')
    ax.bar(x - 0.5*w, col_minade_base, w, label='ColRate_minADE (baseline)',       color='#e0a870')
ax.bar(x + 0.5*w, col_avg_ours,    w, label=f'ColRate_avg ({COLLISION_LOSS})',    color='#5599dd')
ax.bar(x + 1.5*w, col_minade_ours, w, label=f'ColRate_minADE ({COLLISION_LOSS})', color='#55bbaa')

ax.set_xticks(x)
ax.set_xticklabels([s.upper() for s in scenes_with_data])
ax.set_ylabel('Collision rate')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out_fig = f'{SAVE_DIR}/collision_analysis_{COLLISION_LOSS}.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_fig}')

## 8. Collision Trajectory Visualization

Shows scenes where the model still predicts colliding trajectories.

- **Coloured hull** -- convex hull of all K=20 sampled paths per ped; overlapping = collision zone
- **Dark red lines** -- individual colliding samples
- **Orange dashed** -- min-ADE (best) sample
- **Solid + square** -- observed past; **dashed + star** -- ground truth future
- **Red X** -- collision midpoint on best sample
- Title shows ped IDs and frame range for cross-model CSV comparison

In [ ]:
import os, pickle, copy, random
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch, numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import torch.distributions.multivariate_normal as torchdist
from torch.utils.data import DataLoader
from scipy.spatial import ConvexHull

from utils import TrajectoryDataset
from metrics import seq_to_nodes, nodes_rel_to_nodes_abs
from model import social_stgcnn

VIZ_DATASETS = DATASETS
MAX_SCENES   = 3
MAX_PEDS     = 6
SHOW_COL     = 5

PED_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd',
              '#8c564b', '#e377c2', '#17becf', '#bcbd22']


def _has_collision(traj, d=D_COL):
    T, N, _ = traj.shape
    for i in range(N):
        for j in range(i + 1, N):
            if np.any(np.linalg.norm(traj[:, i] - traj[:, j], axis=-1) < d):
                return True
    return False


def _collision_midpoints(traj, d=D_COL):
    T, N, _ = traj.shape
    pts = []
    for i in range(N):
        for j in range(i + 1, N):
            dists = np.linalg.norm(traj[:, i] - traj[:, j], axis=-1)
            for t in np.where(dists < d)[0]:
                pts.append((traj[t, i] + traj[t, j]) / 2)
    return pts


def collect_scenes(exp_path, max_n=MAX_SCENES):
    with open(exp_path + '/args.pkl', 'rb') as f:
        args = pickle.load(f)

    dset = TrajectoryDataset(
        f'./datasets/{args.dataset}/test/',
        obs_len=args.obs_seq_len, pred_len=args.pred_seq_len,
        skip=1, norm_lap_matr=True)
    loader = DataLoader(dset, batch_size=1, shuffle=False, num_workers=0)

    model = social_stgcnn(
        n_stgcnn=args.n_stgcnn, n_txpcnn=args.n_txpcnn,
        output_feat=args.output_size, seq_len=args.obs_seq_len,
        kernel_size=args.kernel_size, pred_seq_len=args.pred_seq_len,
    ).to(device)
    model.load_state_dict(torch.load(exp_path + '/val_best.pth', map_location=device))
    model.eval()

    pool_best_col, pool_any_col = [], []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if len(pool_best_col) >= max_n*3 and len(pool_any_col) >= max_n*3:
                break
            batch = [t.to(device) for t in batch]
            obs_traj, _, obs_traj_rel, _, _, _, V_obs, A_obs, V_tr, _ = batch

            N = obs_traj_rel.shape[1]
            if N < 2 or N > MAX_PEDS:
                continue

            ped_ids   = dset.seq_ped_ids[batch_idx]
            frame_ids = dset.seq_frame_ids[batch_idx]

            V_pred, _ = model(V_obs.permute(0, 3, 1, 2), A_obs.squeeze())
            V_pred = V_pred.permute(0, 2, 3, 1).squeeze()[:, :N, :]
            V_tr_s = V_tr.squeeze()[:, :N, :]

            sx   = torch.exp(V_pred[:, :, 2])
            sy   = torch.exp(V_pred[:, :, 3])
            corr = torch.tanh(V_pred[:, :, 4])
            cov  = torch.zeros(*V_pred.shape[:2], 2, 2, device=device)
            cov[:, :, 0, 0] = sx * sx
            cov[:, :, 0, 1] = corr * sx * sy
            cov[:, :, 1, 0] = corr * sx * sy
            cov[:, :, 1, 1] = sy * sy
            mvn = torchdist.MultivariateNormal(V_pred[:, :, 0:2], cov)

            V_x   = seq_to_nodes(obs_traj.cpu().numpy().copy())
            obs_a = nodes_rel_to_nodes_abs(
                V_obs.cpu().numpy().squeeze().copy(), V_x[0].copy())
            gt_a  = nodes_rel_to_nodes_abs(
                V_tr_s.cpu().numpy().squeeze().copy(), V_x[-1].copy())

            samples, ade_s = [], []
            for _ in range(KSTEPS):
                s  = mvn.sample()
                sa = nodes_rel_to_nodes_abs(
                    s.cpu().numpy().squeeze().copy(), V_x[-1].copy())
                samples.append(copy.deepcopy(sa))
                ade_s.append(float(np.mean(np.sqrt(np.sum((sa - gt_a)**2, axis=-1)))))

            colliding = [_has_collision(s) for s in samples]
            if not any(colliding):
                continue

            best_idx = int(np.argmin(ade_s))
            entry = {
                'obs': obs_a, 'gt': gt_a,
                'samples': samples, 'colliding': colliding,
                'best_idx': best_idx, 'N': N, 'ds': args.dataset,
                'ped_ids': ped_ids, 'frame_ids': frame_ids, 'scene_idx': batch_idx,
            }
            if colliding[best_idx]:
                pool_best_col.append(entry)
            else:
                pool_any_col.append(entry)

    return (pool_best_col + pool_any_col)[:max_n]


def plot_scene(sc, ax, title=''):
    obs, gt   = sc['obs'], sc['gt']
    samples   = sc['samples']
    colliding = sc['colliding']
    best_idx  = sc['best_idx']
    N         = sc['N']

    # Convex hull per pedestrian over all K*T sampled points
    for n in range(N):
        c = PED_COLORS[n % len(PED_COLORS)]
        all_pts = np.concatenate(
            [samples[k][:, n, :] for k in range(len(samples))], axis=0)
        if len(all_pts) >= 3:
            try:
                hull = ConvexHull(all_pts)
                verts = hull.vertices
                ax.fill(all_pts[verts, 0], all_pts[verts, 1],
                        alpha=0.13, color=c, zorder=1)
                closed = np.append(verts, verts[0])
                ax.plot(all_pts[closed, 0], all_pts[closed, 1],
                        color=c, alpha=0.35, linewidth=0.8, zorder=2)
            except Exception:
                pass

    # A few colliding sample lines
    col_idx  = [k for k, c in enumerate(colliding) if c]
    show_col = random.sample(col_idx, min(SHOW_COL, len(col_idx)))
    for k in show_col:
        for n in range(N):
            xs = np.r_[obs[-1, n, 0], samples[k][:, n, 0]]
            ys = np.r_[obs[-1, n, 1], samples[k][:, n, 1]]
            ax.plot(xs, ys, color='#cc2200', alpha=0.30, linewidth=0.9, zorder=3)

    # Min-ADE sample
    best = samples[best_idx]
    for n in range(N):
        xs = np.r_[obs[-1, n, 0], best[:, n, 0]]
        ys = np.r_[obs[-1, n, 1], best[:, n, 1]]
        ax.plot(xs, ys, color='#ff8800', alpha=0.95, linewidth=2.2,
                linestyle='--', zorder=4)

    # Collision X markers
    for pt in _collision_midpoints(best):
        ax.scatter(pt[0], pt[1], marker='x', s=160, color='red',
                   linewidths=3.0, zorder=7)

    # Observed past + GT future
    for n in range(N):
        c = PED_COLORS[n % len(PED_COLORS)]
        ax.plot(obs[:, n, 0], obs[:, n, 1], '-o', color=c,
                linewidth=2, markersize=3, zorder=5)
        ax.scatter(obs[0, n, 0], obs[0, n, 1], marker='s', s=50,
                   color=c, edgecolors='k', linewidths=0.8, zorder=8)
        gx = np.r_[obs[-1, n, 0], gt[:, n, 0]]
        gy = np.r_[obs[-1, n, 1], gt[:, n, 1]]
        ax.plot(gx, gy, '--', color=c, linewidth=2, alpha=0.85, zorder=5)
        ax.scatter(gt[-1, n, 0], gt[-1, n, 1], marker='*', s=120,
                   color=c, edgecolors='k', linewidths=0.8, zorder=8)

    n_col      = sum(colliding)
    best_tag   = '[best collides]' if colliding[best_idx] else '[best clean]'
    ped_id_str = 'peds: ' + ', '.join(str(int(p)) for p in sc['ped_ids'])
    obs_len_sc = obs.shape[0]
    frame_ids  = sc['frame_ids']
    frame_str  = (f'frames {int(frame_ids[0])}-{int(frame_ids[obs_len_sc-1])} obs'
                  f' | {int(frame_ids[obs_len_sc])}-{int(frame_ids[-1])} pred')
    ax.set_title(
        f'{title}\n{n_col}/{KSTEPS} collide  |  {best_tag}\n'
        f'{ped_id_str}  |  {frame_str}',
        fontsize=8.0
    )
    ax.set_aspect('equal', adjustable='datalim')
    ax.grid(True, alpha=0.25)
    ax.set_xlabel('x (m)', fontsize=8)
    ax.set_ylabel('y (m)', fontsize=8)


# Collect
random.seed(42)
all_sc = {}
for ds in VIZ_DATASETS:
    ep = f'{SAVE_DIR}/social-stgcnn-{ds}'
    if not os.path.isfile(ep + '/val_best.pth'):
        print(f'SKIP {ds} -- checkpoint missing'); continue
    print(f'Scanning {ds}...')
    all_sc[ds] = collect_scenes(ep)
    n_best = sum(sc['colliding'][sc['best_idx']] for sc in all_sc[ds])
    print(f'  -> {len(all_sc[ds])} scenes  ({n_best} with best-sample collision)')

# Plot
viz_ds = [ds for ds in VIZ_DATASETS if ds in all_sc and all_sc[ds]]

if not viz_ds:
    print('No collision scenes found -- the model may have learned to avoid all collisions!')
else:
    nrows = len(viz_ds)
    ncols = MAX_SCENES
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5 * ncols, 5.8 * nrows), squeeze=False)
    for ri, ds in enumerate(viz_ds):
        for ci, sc in enumerate(all_sc[ds]):
            plot_scene(sc, axes[ri, ci],
                       title=f'{ds.upper()} -- scene {ci+1}  ({sc["N"]} peds)')
        for ci in range(len(all_sc[ds]), ncols):
            axes[ri, ci].set_visible(False)

    legend_handles = [
        mpatches.Patch(color='gray', alpha=0.25,
                       label='Sampled region (convex hull per ped)\nOverlapping = potential collision zone'),
        mlines.Line2D([], [], color='#cc2200', alpha=0.6, lw=1.2,
                      label=f'Colliding samples (<={SHOW_COL} shown)'),
        mlines.Line2D([], [], color='#ff8800', lw=2.2, ls='--',
                      label='min-ADE sample (best)'),
        mlines.Line2D([], [], color='gray', lw=2, ls='-', marker='o',
                      markersize=4, label='Observed past'),
        mlines.Line2D([], [], color='gray', lw=2, ls='--',
                      label='Ground truth future'),
        mlines.Line2D([], [], color='red', lw=0, marker='x',
                      markersize=10, markeredgewidth=3,
                      label=f'Collision point on best sample (d<{D_COL}m)'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=3,
               fontsize=9, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(
        f'Social-STGCNN ({COLLISION_LOSS} loss) -- Collision Trajectory Visualization\n'
        f'(K={KSTEPS} samples, d_col={D_COL} m)',
        fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0.10, 1, 1])
    out_fig = f'{SAVE_DIR}/collision_visualization_{COLLISION_LOSS}.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_fig}')

## 9. CSV export

One row per (dataset x scene x pedestrian).  
Cross-model comparison:
```python
import pandas as pd
df = pd.concat([
    pd.read_csv('scene_ped_metadata.csv'),
    pd.read_csv('scene_ped_metadata_hinge.csv'),
])
df.groupby(['dataset','scene_num','ped_id'])[['n_colliding_samples','best_sample_collides']].first()
```

In [ ]:
import pandas as pd
import os
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_LABEL = COLLISION_LOSS
CSV_PATH    = f'{SAVE_DIR}/scene_ped_metadata_{COLLISION_LOSS}.csv'

rows = []
for ds, scenes in all_sc.items():
    for scene_num, sc in enumerate(scenes, start=1):
        frame_ids    = sc['frame_ids']
        obs_len_sc   = sc['obs'].shape[0]
        obs_f_start  = int(frame_ids[0])
        obs_f_end    = int(frame_ids[obs_len_sc - 1])
        pred_f_start = int(frame_ids[obs_len_sc])
        pred_f_end   = int(frame_ids[-1])
        all_frames   = ';'.join(str(int(f)) for f in frame_ids)
        n_col        = sum(sc['colliding'])
        best_col     = int(sc['colliding'][sc['best_idx']])
        for ped_id in sc['ped_ids']:
            rows.append({
                'model':                MODEL_LABEL,
                'dataset':              ds,
                'scene_num':            scene_num,
                'scene_idx':            sc['scene_idx'],
                'ped_id':               int(ped_id),
                'obs_frame_start':      obs_f_start,
                'obs_frame_end':        obs_f_end,
                'pred_frame_start':     pred_f_start,
                'pred_frame_end':       pred_f_end,
                'all_frame_ids':        all_frames,
                'n_peds':               sc['N'],
                'n_colliding_samples':  n_col,
                'best_sample_collides': best_col,
            })

df_meta = pd.DataFrame(rows)
df_meta.to_csv(f'{SAVE_DIR}/{CSV_PATH}', index=False)
print(f'Saved {len(rows)} rows -> {CSV_PATH}')
print(df_meta.to_string(index=False))